# 911 Calls Wav2Vec2
https://www.kaggle.com/code/ajax0564/video-caption-using-wavetovec-2-0<br/>
https://www.kaggle.com/code/stpeteishii/english-audio-wav2vec2

A "911 call" refers to a telephone call made to the emergency telephone number 911 in the United States and several other countries. The number 911 is a universal emergency number that allows people to quickly reach emergency services, such as police, fire departments, and medical services. When someone dials 911, their call is routed to a local emergency dispatch center, also known as a Public Safety Answering Point (PSAP), where trained operators or dispatchers assess the situation and send appropriate help to the caller's location.

## facebook/wav2vec2-base-960h
The facebook/wav2vec2-base-960h is a speech recognition model from Facebook AI, trained on a massive dataset of unlabeled speech audio. It is a Transformer-based model with 960 hidden units, and it can be used to transcribe speech into text.

The model was pre-trained on 53,000 hours of unlabeled speech audio from the Librispeech dataset. It was then fine-tuned on 10 minutes of labeled speech audio from the same dataset. This results in a model that can achieve a word error rate (WER) of 4.8% on the Librispeech test set.

The facebook/wav2vec2-base-960h model can be used for a variety of speech recognition tasks, such as:

Transcribing audio recordings of meetings or lectures
Translating audio recordings of foreign language speech
Creating closed captioning for videos
Developing virtual assistants that can understand spoken commands
The model is available on the Hugging Face Hub, and it can be used with a variety of frameworks, such as PyTorch and TensorFlow.

In [1]:
!pip install torch==1.13.0

ERROR: Could not find a version that satisfies the requirement torch==1.13.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1)
ERROR: No matching distribution found for torch==1.13.0


In [2]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 84.9 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.5 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.7 MB/s eta 0:00:0000:010:01
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=ed3a72065147e26f22fca2a03a22b291add630aae19e389ba340d8b8c337e9a7
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 

In [4]:
import os
for f in os.listdir("/kaggle/input/datasets/bond005/openai-whisper-small"):
    print(f)

config.json
merges.txt
preprocessor_config.json
README.md
tokenizer.json
vocab.json
normalizer.json
tokenizer_config.json
pytorch_model.bin
special_tokens_map.json
added_tokens.json
generation_config.json


In [5]:
import os
for d in os.listdir("/kaggle/input"):
    print(d)

911-recordings-first-6-seconds
datasets


In [6]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import librosa

MODEL_PATH = "/kaggle/input/datasets/bond005/openai-whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH)
model.eval()

print("Loaded successfully!")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loaded successfully!


In [7]:
import os
for root, dirs, files in os.walk("/kaggle/input/911-recordings-first-6-seconds"):
    for f in files[:5]:  # just show first 5 files
        print(os.path.join(root, f))

/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_650_0.wav
/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_15_0.wav
/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_389_0.wav
/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_433_0.wav
/kaggle/input/911-recordings-first-6-seconds/911_first6sec/call_560_1.wav


In [9]:
import os
import torch
import librosa
import pandas as pd
import spacy

!pip install spacy
!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

AUDIO_DIR = "/kaggle/input/911-recordings-first-6-seconds/911_first6sec"
MODEL_PATH = "/kaggle/input/datasets/bond005/openai-whisper-small"

URGENCY_WORDS = ["help", "fire", "dying", "gun", "shot", "crash", "trapped", 
                 "bleeding", "emergency", "please", "hurt", "dead", "killed",
                 "kill", "murder", "suicide", "pills", "shooting", "accident",
                 "fell", "unconscious", "not responding", "blood"]

def transcribe(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="english", task="transcribe")
    with torch.no_grad():
        predicted_ids = model.generate(
            inputs["input_features"],
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=100
        )
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

def extract_event(text):
    keywords = {
        "fire": "Fire", "shot": "Shooting", "gun": "Shooting",
        "crash": "Traffic Accident", "accident": "Traffic Accident",
        "robbery": "Robbery", "theft": "Theft", "fight": "Assault",
        "hurt": "Injury", "bleeding": "Medical Emergency",
        "dead": "Homicide", "kill": "Homicide", "murder": "Homicide",
        "suicide": "Suicide", "pills": "Medical Emergency",
        "unconscious": "Medical Emergency", "blood": "Medical Emergency"
    }
    text_lower = text.lower()
    for k, v in keywords.items():
        if k in text_lower:
            return v
    return "Unknown"

def urgency_score(text):
    text_lower = text.lower()
    hits = sum(1 for w in URGENCY_WORDS if w in text_lower)
    return round(min(hits / 3, 1.0), 2)  # lower threshold for short 6-sec clips

def sentiment_label(score):
    if score >= 0.6:
        return "Distressed"
    elif score >= 0.2:
        return "Concerned"
    return "Calm"

def extract_location(text):
    doc = nlp(text)
    locs = [ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC", "FAC")]
    return locs[0] if locs else "Unknown"

# Get first 20 audio files
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")][:20]

results = []
for i, fname in enumerate(audio_files):
    path = os.path.join(AUDIO_DIR, fname)
    print(f"Processing {i+1}/{len(audio_files)}: {fname}")
    transcript = transcribe(path)
    score = urgency_score(transcript)
    results.append({
        "Call_ID": f"AUD-{str(i+1).zfill(3)}",
        "Transcript": transcript,
        "Extracted_Event": extract_event(transcript),
        "Location": extract_location(transcript),
        "Sentiment": sentiment_label(score),
        "Urgency_Score": score
    })

df = pd.DataFrame(results)
df.to_csv("/kaggle/working/audio_output.csv", index=False)
print("\nDone! Here's a preview:")
display(df)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 80.2 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Processing 1/20: call_650_0.wav


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing 2/20: call_15_0.wav


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing 3/20: call_389_0.wav


...

[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing 20/20: call_36_0.wav


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Done! Here's a preview:


,Call_ID,Transcript,Extracted_Event,Location,Sentiment,Urgency_Score
0,AUD-001,I just killed my daughter. You what? I just k...,Homicide,Unknown,Distressed,0.67
1,AUD-002,I'm Dillon Pearson and I just killed two people.,Homicide,Unknown,Distressed,0.67
2,AUD-003,Hello?,Unknown,Unknown,Calm,0.00
3,AUD-004,How you doing? I'd like to report like tumor ...,Homicide,Unknown,Concerned,0.33
4,AUD-005,So we're aware of it. OK.,Unknown,Unknown,Calm,0.00
5,AUD-006,I'll play just crashed. Okay. You said three ...,Traffic Accident,Unknown,Concerned,0.33
6,AUD-007,"Um, I took some pills and the pills made me g...",Medical Emergency,Unknown,Concerned,0.33
7,AUD-008,"Well, I just got a phone call saying that my ...",Unknown,Unknown,Calm,0.00
8,AUD-009,"Yes, I'm staring at my bed in the back. I und...",Unknown,Unknown,Calm,0.00
9,AUD-010,"Each man, I believe, may have a possible, hav...",Unknown,Unknown,Calm,0.00
